# Debug a CrewAI workflow with Maida

This tutorial drives CrewAI's public execution-hook API with a deterministic quarterly-sales workflow. It exercises the exact hook boundary used by real crews while staying offline: there are no API keys and no network calls.

## Setup

Install Maida with CrewAI support (run once):

In [ ]:
# %pip install --upgrade pip -q
# %pip install "maida-ai[crewai]" -q

CrewAI initializes local state when imported and may enable its own telemetry. The next cell redirects that state to a temporary directory and disables CrewAI telemetry before importing the framework. Maida's local trace remains enabled.

In [ ]:
import os
import tempfile
from types import SimpleNamespace

os.environ.setdefault("CREWAI_STORAGE_DIR", tempfile.mkdtemp(prefix="maida-crewai-"))
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")

import maida
from maida.integrations import crewai as maida_crewai  # importing registers hooks
from crewai.hooks import (
    LLMCallHookContext,
    ToolCallHookContext,
    get_after_llm_call_hooks,
    get_after_tool_call_hooks,
    get_before_llm_call_hooks,
    get_before_tool_call_hooks,
)

## Build a deterministic hook harness

CrewAI invokes registered before/after hooks around each LLM and tool operation. These helpers create the same public hook contexts and call the registered hooks in the same order. The stub context keeps framework metadata realistic without constructing a network-backed model.

In [ ]:
AGENT = SimpleNamespace(role="Sales analyst")
TASK = SimpleNamespace(description="Search, calculate, and save quarterly sales")
CREW = SimpleNamespace()
LLM = SimpleNamespace(model="scripted/offline")


def emit_llm_call(prompt: str, response: str) -> str:
    context = LLMCallHookContext(
        messages=[{"role": "user", "content": prompt}],
        llm=LLM,
        agent=AGENT,
        task=TASK,
        crew=CREW,
    )
    for hook in get_before_llm_call_hooks():
        hook(context)
    context.response = response
    for hook in get_after_llm_call_hooks():
        hook(context)
    return response


def run_tool(name: str, arguments: dict, function):
    context = ToolCallHookContext(
        tool_name=name,
        tool_input=arguments,
        tool=None,
        agent=AGENT,
        task=TASK,
        crew=CREW,
    )
    for hook in get_before_tool_call_hooks():
        hook(context)
    result = function(**arguments)
    context.tool_result = result
    for hook in get_after_tool_call_hooks():
        hook(context)
    return result

## Define deterministic tools

These are the same search → calculate → save operations used by the LangGraph and OpenAI Agents tutorials. The `api_key` argument is deliberately secret-shaped so the stored trace demonstrates Maida's redaction boundary.

In [ ]:
def search(query: str) -> str:
    return "Q1: 1.2M, Q2: 1.5M, Q3: 2.0M, Q4: 1.8M"


def calculator(expression: str) -> str:
    if expression != "1.2+1.5+2.0+1.8":
        raise ValueError(f"Unexpected expression: {expression}")
    return "6.5"


def save_result(data: str, api_key: str) -> str:
    return f"Saved: {data}"

## Run the happy path under Maida

Importing `maida.integrations.crewai` registers Maida's execution hooks. The `@maida.trace` wrapper creates the active run those hooks record into.

In [ ]:
@maida.trace(name="CrewAI hooks (happy path)")
def run_crewai_happy_path():
    emit_llm_call("Get quarterly sales.", "I will search.")
    sales = run_tool("search", {"query": "quarterly sales"}, search)

    emit_llm_call(sales, "I will calculate the total.")
    total = run_tool(
        "calculator",
        {"expression": "1.2+1.5+2.0+1.8"},
        calculator,
    )

    emit_llm_call(total, "I will save the result.")
    saved = run_tool(
        "save_result",
        {"data": f"{total}M total", "api_key": "tutorial-secret"},
        save_result,
    )

    return emit_llm_call(saved, "DONE. Quarterly sales total: 6.5M.")


print(run_crewai_happy_path())
print("Run complete. View with: maida view")

In [ ]:
!maida list

## What happened in this run

The expected structural signature is four `LLM_CALL` events and three `TOOL_CALL` events in this order: search, calculator, and save_result. This matches the equivalent LangGraph and OpenAI Agents happy paths.

Open the `save_result` event in `maida view`: its stored arguments retain the `api_key` field name but replace the value with `__REDACTED__`. Framework-specific context stays in `meta.crewai.*`; the adapter still emits Maida's standard event types.

## Exercise a failure path

If a tool raises after CrewAI's before-hook but before its after-hook, the adapter flushes the pending operation when the Maida run exits. The trace records one error-status `TOOL_CALL`, the Python exception, and an error-status run instead of silently losing the incomplete tool call.

In [ ]:
@maida.trace(name="CrewAI hooks (tool failure)")
def run_crewai_failure_path():
    return run_tool(
        "calculator",
        {"expression": "unsupported"},
        calculator,
    )


try:
    run_crewai_failure_path()
    print("Unexpected: tool did not fail")
except ValueError as error:
    print(f"Expected tool failure: {error}")

## Failure cases and limitations

- **Missing dependency:** importing the adapter without the CrewAI extra raises an `ImportError` with `pip install maida-ai[crewai]` guidance. Install the extra shown in Setup.
- **No active Maida run:** CrewAI hooks intentionally do nothing unless `crew.kickoff()`, `flow.kickoff()`, or a hook harness runs inside `@maida.trace` or `maida.traced_run(...)`.
- **Hook ordering:** CrewAI calls before-hooks in registration order. If an earlier before-hook returns `False`, Maida may not observe that blocked call.
- **Guardrails:** CrewAI's hooks do not provide the same immediate exception-propagation contract as the LangChain handler. Use the recorded trace and your application control flow to enforce termination.
- **Hook harness scope:** this notebook intentionally drives CrewAI's public hook boundary instead of invoking a model-backed `Crew`. The same registered hooks receive contexts from real crews and flows.

### Redaction and truncation

CrewAI prompts, responses, tool arguments, results, and framework metadata pass through Maida's existing storage redaction and truncation paths. Keep redaction enabled (the default) and configure `MAIDA_MAX_FIELD_BYTES` when payloads need a tighter size limit. Redaction protects the stored Maida trace; it does not change what CrewAI sends to an external model provider in a real application.

## Next steps

- Run `maida view` to compare the successful and failed CrewAI traces.
- Compare the happy-path event sequence with the LangGraph and OpenAI Agents notebooks.
- Rerun the happy path, use `maida baseline`, then change a tool path and run `maida assert` to see the behavioral regression gate fail.